# Datathon 2026 — Candidate Generator + CatBoostRanker YetiRank GPU

This notebook keeps the strong rule-based candidate generator, but **does not use the rule-based popularity score as the final reranker**.

Main flow:

1. Build leak-safe candidate pools at each cutoff.
2. Build train/valid labels from future positive interactions.
3. Generate top-150 candidates per user using the existing rule retrieval logic.
4. Train `CatBoostRanker(loss_function="YetiRank")` on GPU.
5. Validate with `Recall@10` and `NDCG@10`.
6. Final prediction:
   - warm users: CatBoost YetiRank rerank
   - true-cold users: use `submission2.csv` fallback

Expected final output:

`/kaggle/working/submission_yetirank_gpu.csv`

In [1]:
# ============================================================
# CELL 1: IMPORTS + GLOBAL CONFIG
# ============================================================

import os
import gc
import csv
import math
import glob
import json
import hashlib
import warnings
from pathlib import Path
from collections import defaultdict, Counter
from datetime import timedelta

import numpy as np
import pandas as pd
import polars as pl

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(30)

try:
    from catboost import CatBoostRanker, Pool
    CATBOOST_OK = True
except Exception as e:
    CATBOOST_OK = False
    print("CatBoost import failed:", repr(e))

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# -------------------- Main knobs --------------------
TOPN = 10
CANDIDATE_TOPK = 150
HISTORY_DAYS = 31
SELLER_CAP = 3

# Candidate retrieval pool before taking top150.
PREFILTER_PER_BUCKET = 350
MAX_PREFILTER_CANDIDATES = 3000

# To avoid OOM during YetiRank training.
# Increase only if memory is stable.
TRAIN_USER_LIMIT = 120_000       # None = all train label users
VALID_USER_LIMIT = 50_000        # None = all valid label users
MAX_NEG_PER_USER = 40            # label 0 sampled per user; positives kept if candidate contains them
RANDOM_SEED = 42

# If True, rule popularity score is NOT used as a numeric feature.
# We keep candidate rank/source-like/context features instead.
DROP_RULE_SCORE_FEATURE = True

# Final cold fallback from a strong rule-based/public file.
# Upload submission2.csv to Kaggle input, or put it in /kaggle/working.
SUBMISSION2_CANDIDATES = [
    "/kaggle/input/datasets/nvap2011/submission2/submission2.csv",
    "/kaggle/input/*/submission2.csv",
    "/kaggle/working/submission2.csv",
]

FINAL_OUTPUT_PATH = WORK_DIR / "submission_yetirank_gpu.csv"
MODEL_PATH = WORK_DIR / "catboost_yetirank_gpu.cbm"

# Time-based split. Final target is 2026-04-10 -> 2026-05-07.
TRAIN_CUTOFF = pd.Timestamp("2026-02-12")
TRAIN_LABEL_START = pd.Timestamp("2026-02-13")
TRAIN_LABEL_END = pd.Timestamp("2026-03-12")

VALID_CUTOFF = pd.Timestamp("2026-03-12")
VALID_LABEL_START = pd.Timestamp("2026-03-13")
VALID_LABEL_END = pd.Timestamp("2026-04-09")

FINAL_CUTOFF = pd.Timestamp("2026-04-09")

POSITIVE_EVENTS = ["view_phone", "contact_chat", "contact_zalo", "contact_sms", "other_interaction"]

EVENT_WEIGHT = {
    "pageview": 1.00,
    "other_interaction": 1.25,
    "view_phone": 6.00,
    "contact_chat": 7.00,
    "contact_zalo": 8.00,
    "contact_sms": 7.00,
}

LET_PRICE_ORDER = [
    "<2M/tháng", "2M-3M/tháng", "3M-5M/tháng", "5M-7M/tháng",
    "7M-10M/tháng", "10M-15M/tháng", "15M-20M/tháng",
    "20M-30M/tháng", ">30M/tháng"
]
SELL_PRICE_ORDER = [
    "<500M", "500M-800M", "800M-1B", "1B-1.5B", "1.5B-2B",
    "2B-3B", "3B-5B", "5B-7B", "7B-10B", "10B-15B",
    "15B-20B", ">20B"
]

rng = np.random.default_rng(RANDOM_SEED)

print("WORK_DIR:", WORK_DIR)
print("CATBOOST_OK:", CATBOOST_OK)
print("CANDIDATE_TOPK:", CANDIDATE_TOPK)
print("MAX_NEG_PER_USER:", MAX_NEG_PER_USER)
print("DROP_RULE_SCORE_FEATURE:", DROP_RULE_SCORE_FEATURE)

WORK_DIR: /kaggle/working
CATBOOST_OK: True
CANDIDATE_TOPK: 150
MAX_NEG_PER_USER: 40
DROP_RULE_SCORE_FEATURE: True


In [2]:
# ============================================================
# CELL 2: FIND DATASET PATHS + COMMON HELPERS
# ============================================================

def find_dataset_dirs():
    roots = [Path("/kaggle/input"), Path(".")]
    data1 = None
    data2 = None

    # Common explicit paths first.
    candidates1 = [
        Path("/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1"),
        Path("/kaggle/input/datasets/nguyenminhquanhcmus/datathon2026-1"),
        Path("/kaggle/input/Datathon2026_1"),
        Path("/kaggle/input/datathon2026-1"),
    ]
    candidates2 = [
        Path("/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2"),
        Path("/kaggle/input/datasets/nguyenminhquanhcmus/datathon2026-2"),
        Path("/kaggle/input/Datathon2026_2"),
        Path("/kaggle/input/datathon2026-2"),
    ]
    for p in candidates1:
        if data1 is None and (p / "dim_listing").exists():
            data1 = p
    for p in candidates2:
        if data2 is None and (p / "fact_user_events").exists():
            data2 = p

    if data1 is None or data2 is None:
        for root in roots:
            if not root.exists():
                continue
            for p in root.rglob("*"):
                if not p.is_dir():
                    continue
                if data1 is None and (p / "dim_listing").exists() and (p / "test" / "test_users.parquet").exists():
                    data1 = p
                if data2 is None and (p / "fact_user_events").exists():
                    data2 = p
                if data1 is not None and data2 is not None:
                    break
            if data1 is not None and data2 is not None:
                break

    if data1 is None or data2 is None:
        raise FileNotFoundError("Cannot find datathon2026-1 / datathon2026-2 folders under /kaggle/input.")
    return data1, data2

DATA1_DIR, DATA2_DIR = find_dataset_dirs()
DIM_FILES = sorted((DATA1_DIR / "dim_listing").glob("*.parquet"))
SNAPSHOT_FILES = sorted((DATA1_DIR / "fact_listing_snapshot").glob("*.parquet"))
EVENT_FILES = sorted((DATA2_DIR / "fact_user_events").glob("*.parquet"))
TEST_USERS_PATH = DATA1_DIR / "test" / "test_users.parquet"

print("DATA1_DIR:", DATA1_DIR)
print("DATA2_DIR:", DATA2_DIR)
print("n_dim_files:", len(DIM_FILES))
print("n_snapshot_files:", len(SNAPSHOT_FILES))
print("n_event_files:", len(EVENT_FILES))
print("TEST_USERS_PATH:", TEST_USERS_PATH)

# Price lookup.
price_rows = []
for i, bucket in enumerate(LET_PRICE_ORDER):
    price_rows.append({"ad_type": "let", "price_bucket": bucket, "price_idx": i})
for i, bucket in enumerate(SELL_PRICE_ORDER):
    price_rows.append({"ad_type": "sell", "price_bucket": bucket, "price_idx": i})
price_lookup_pl = pl.DataFrame(price_rows).with_columns([
    pl.col("ad_type").cast(pl.Utf8),
    pl.col("price_bucket").cast(pl.Utf8),
    pl.col("price_idx").cast(pl.Int16),
])

# Load test users.
test_users_pl = (
    pl.read_parquet(str(TEST_USERS_PATH))
    .select("user_id")
    .with_row_index("test_order")
)
test_user_list = test_users_pl["user_id"].cast(pl.Utf8).to_list()
test_user_set = set(test_user_list)
print("n_test_users:", len(test_user_list))
display(test_users_pl.head())

def tag_date(ts):
    return pd.Timestamp(ts).strftime("%Y%m%d")

def ensure_str_cols(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = df[c].fillna("__missing__").astype(str)
    return df

def safe_round_bin(series, step, missing=-1, dtype="int32"):
    x = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    b = (np.round(x / step) * step).where(x.notna(), missing)
    return b.fillna(missing).astype(dtype)

def hash_sample_users(users, limit=None, seed=RANDOM_SEED):
    users = list(map(str, users))
    if limit is None or len(users) <= limit:
        return users
    # deterministic random sample
    r = np.random.default_rng(seed)
    idx = r.choice(len(users), size=limit, replace=False)
    return [users[i] for i in idx]

def find_submission2_path():
    paths = []
    for pat in SUBMISSION2_CANDIDATES:
        paths.extend(glob.glob(pat))
    paths = [Path(p) for p in paths if Path(p).exists()]
    if not paths:
        print("[WARN] submission2.csv not found. Cold fallback will use global candidate popularity.")
        return None
    # Prefer shortest/explicit path.
    paths = sorted(paths, key=lambda p: (len(str(p)), str(p)))
    print("SUBMISSION2_PATH:", paths[0])
    return paths[0]

SUBMISSION2_PATH = find_submission2_path()

DATA1_DIR: /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1
DATA2_DIR: /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2
n_dim_files: 40
n_snapshot_files: 62
n_event_files: 500
TEST_USERS_PATH: /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1/test/test_users.parquet
n_test_users: 161568


test_order,user_id
u32,str
0,"""c9911d7e40a8c8b7ffd257e21177fc…"
1,"""612e00627e215ac3086d919c6a9dc5…"
2,"""f1b4015a5dc2884da1152fc097642b…"
3,"""bab7641e6e2a625a6a3ae924f168a7…"
4,"""0f76d272d5d06e84d34cbee06d9228…"


SUBMISSION2_PATH: /kaggle/input/datasets/nvap2011/submission2/submission2.csv


In [3]:
# ============================================================
# CELL 3: BUILD LEAK-SAFE CANDIDATE POOL AT A CUTOFF
# ============================================================

DIM_SELECT_COLS = [
    "item_id", "seller_id", "category", "seller_type", "ad_type",
    "area_sqm", "bedrooms", "bathrooms", "floors", "width_m",
    "direction", "legal_status", "house_type", "furnishing",
    "city_name", "district_name", "ward_name", "project_id",
    "price_bucket", "images_count", "posted_date", "expected_expired_date",
]

SNAP_SELECT_COLS = ["item_id", "date", "views_24h", "contacts_24h", "listing_age_days"]

def build_candidate_pool(cutoff, name):
    cutoff = pd.Timestamp(cutoff).date()
    out_path = WORK_DIR / f"candidate_pool_{name}_{tag_date(cutoff)}.parquet"
    if out_path.exists():
        cand = pd.read_parquet(out_path)
        print(f"[LOAD] candidate pool {name}:", cand.shape, out_path)
        return cleanup_candidate_df(cand)

    print("=" * 100)
    print(f"[BUILD CANDIDATE POOL] name={name} cutoff={cutoff}")
    print("=" * 100)

    dim_lf = (
        pl.scan_parquet([str(p) for p in DIM_FILES])
        .select([c for c in DIM_SELECT_COLS])
        .with_columns([
            pl.col("item_id").cast(pl.Utf8),
            pl.col("seller_id").cast(pl.Utf8),
            pl.col("posted_date").cast(pl.Date, strict=False),
            pl.col("expected_expired_date").cast(pl.Date, strict=False),
            pl.col("ad_type").cast(pl.Utf8),
            pl.col("price_bucket").cast(pl.Utf8),
        ])
        .filter(pl.col("ad_type").is_in(["let", "sell"]))
        .filter(pl.col("posted_date").is_null() | (pl.col("posted_date") <= pl.lit(cutoff)))
        .filter(pl.col("expected_expired_date").is_null() | (pl.col("expected_expired_date") >= pl.lit(cutoff)))
    )

    snap_lf = (
        pl.scan_parquet([str(p) for p in SNAPSHOT_FILES])
        .select([c for c in SNAP_SELECT_COLS])
        .with_columns([
            pl.col("item_id").cast(pl.Utf8),
            pl.col("date").cast(pl.Date, strict=False),
        ])
        .filter(pl.col("date") <= pl.lit(cutoff))
    )

    pop_start = cutoff - timedelta(days=30)
    pop_lf = (
        snap_lf
        .filter(pl.col("date") >= pl.lit(pop_start))
        .group_by("item_id")
        .agg([
            pl.sum("views_24h").fill_null(0).alias("views_30d"),
            pl.sum("contacts_24h").fill_null(0).alias("contacts_30d"),
            pl.max("date").alias("last_snapshot_date"),
            pl.min("listing_age_days").alias("min_listing_age_days"),
        ])
    )

    candidate_lf = (
        dim_lf
        .join(price_lookup_pl.lazy(), on=["ad_type", "price_bucket"], how="left")
        .join(pop_lf, on="item_id", how="left")
        .with_columns([
            pl.col("views_30d").fill_null(0.0),
            pl.col("contacts_30d").fill_null(0.0),
            pl.col("images_count").fill_null(0.0),
            pl.col("area_sqm").cast(pl.Float32, strict=False),
            pl.col("bedrooms").cast(pl.Float32, strict=False),
            pl.col("bathrooms").cast(pl.Float32, strict=False),
            pl.col("floors").cast(pl.Float32, strict=False),
            pl.col("width_m").cast(pl.Float32, strict=False),
        ])
    )

    cand = candidate_lf.collect(engine="streaming").to_pandas()
    print("active candidate rows collected:", cand.shape)

    cand = cleanup_candidate_df(cand, do_dedup=True)
    cand.to_parquet(out_path, index=False)
    print("[SAVE] candidate pool:", out_path, cand.shape)
    return cand


def cleanup_candidate_df(cand, do_dedup=False):
    cand = cand.copy()
    cand["item_id"] = cand["item_id"].astype(str)

    num_cols = ["area_sqm", "bedrooms", "bathrooms", "floors", "width_m", "images_count", "views_30d", "contacts_30d"]
    for col in num_cols:
        if col in cand.columns:
            cand[col] = pd.to_numeric(cand[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        else:
            cand[col] = np.nan

    cand["price_idx"] = pd.to_numeric(cand.get("price_idx", -1), errors="coerce").fillna(-1).astype("int16")
    cand["category"] = pd.to_numeric(cand.get("category", -1), errors="coerce").fillna(-1).astype("int16")

    str_cols = [
        "seller_id", "seller_type", "ad_type", "city_name", "district_name", "ward_name",
        "project_id", "price_bucket", "direction", "legal_status", "house_type", "furnishing"
    ]
    cand = ensure_str_cols(cand, str_cols)

    cand["area_bin"] = safe_round_bin(cand["area_sqm"], 5)
    cand["bedrooms_bin"] = pd.to_numeric(cand["bedrooms"], errors="coerce").fillna(-1).round().astype("int16")
    cand["bathrooms_bin"] = pd.to_numeric(cand["bathrooms"], errors="coerce").fillna(-1).round().astype("int16")
    cand["floors_bin"] = pd.to_numeric(cand["floors"], errors="coerce").fillna(-1).round().astype("int16")

    cand["seller_private_priority"] = np.where(cand["seller_type"].eq("private"), 0, 1).astype("int8")
    cand["det_rand"] = cand["item_id"].map(lambda x: int(hashlib.md5(str(x).encode()).hexdigest()[:8], 16))

    cand["base_score"] = (
        np.log1p(cand["views_30d"].fillna(0).astype(float))
        + 4.0 * np.log1p(cand["contacts_30d"].fillna(0).astype(float))
        + 0.20 * (cand["seller_type"].eq("private")).astype(float)
        + 0.05 * np.log1p(cand["images_count"].fillna(0).clip(lower=0).astype(float))
    ).astype("float32")

    if do_dedup:
        dedup_cols = [
            "ad_type", "category", "city_name", "district_name", "ward_name",
            "project_id", "price_bucket", "area_bin", "bedrooms_bin", "bathrooms_bin",
            "floors_bin", "house_type", "legal_status"
        ]
        before = len(cand)
        cand = (
            cand
            .sort_values(
                ["seller_private_priority", "base_score", "contacts_30d", "views_30d", "det_rand"],
                ascending=[True, False, False, False, True],
                kind="mergesort"
            )
            .drop_duplicates(dedup_cols, keep="first")
            .reset_index(drop=True)
        )
        print(f"candidate pool after property dedup: {before:,} -> {len(cand):,}")

    cand = cand.sort_values("base_score", ascending=False, kind="mergesort").reset_index(drop=True)
    return cand

In [4]:
# ============================================================
# CELL 4: BUILD LABELS FROM FUTURE POSITIVE INTERACTIONS
# ============================================================

def collect_positive_pairs(label_start, label_end, cutoff, name):
    label_start = pd.Timestamp(label_start)
    label_end = pd.Timestamp(label_end)
    cutoff = pd.Timestamp(cutoff)
    out_path = WORK_DIR / f"positive_pairs_{name}_{tag_date(label_start)}_{tag_date(label_end)}.parquet"
    if out_path.exists():
        df = pd.read_parquet(out_path)
        print(f"[LOAD] labels {name}:", df.shape, out_path)
        return df

    print("=" * 100)
    print(f"[COLLECT LABELS] {name}: {label_start.date()} -> {label_end.date()} | cutoff={cutoff.date()}")
    print("=" * 100)

    parts = []
    start_dt = label_start.to_pydatetime()
    # include whole end day
    end_exclusive = (label_end + pd.Timedelta(days=1)).to_pydatetime()

    for i, f in enumerate(EVENT_FILES, 1):
        try:
            part = (
                pl.scan_parquet(str(f))
                .select([
                    pl.col("user_id").cast(pl.Utf8),
                    pl.col("item_id").cast(pl.Utf8),
                    pl.col("event_type").cast(pl.Utf8),
                    pl.col("event_ts").cast(pl.Datetime, strict=False),
                ])
                .filter(pl.col("event_type").is_in(POSITIVE_EVENTS))
                .filter((pl.col("event_ts") >= pl.lit(start_dt)) & (pl.col("event_ts") < pl.lit(end_exclusive)))
                .select(["user_id", "item_id"])
                .unique()
                .collect(engine="streaming")
            )
            if part.height:
                parts.append(part)
        except Exception as e:
            print("[WARN] label file failed:", f, repr(e))

        if i % 50 == 0:
            print(f"processed event files: {i}/{len(EVENT_FILES)} | parts={len(parts)}")

    if not parts:
        df = pd.DataFrame(columns=["user_id", "item_id"])
    else:
        df = pl.concat(parts).unique().to_pandas()

    # Leak-safe item filter: label item must have been posted before/equal cutoff.
    dim_active = (
        pl.scan_parquet([str(p) for p in DIM_FILES])
        .select([
            pl.col("item_id").cast(pl.Utf8),
            pl.col("posted_date").cast(pl.Date, strict=False),
        ])
        .filter(pl.col("posted_date").is_null() | (pl.col("posted_date") <= pl.lit(cutoff.date())))
        .select("item_id")
        .unique()
        .collect(engine="streaming")
        .to_pandas()
    )
    before = len(df)
    df = df.merge(dim_active, on="item_id", how="inner").drop_duplicates(["user_id", "item_id"])
    print(f"labels after posted_date<=cutoff filter: {before:,} -> {len(df):,}")

    df.to_parquet(out_path, index=False)
    print("[SAVE] labels:", out_path)
    return df

train_labels = collect_positive_pairs(TRAIN_LABEL_START, TRAIN_LABEL_END, TRAIN_CUTOFF, "train")
valid_labels = collect_positive_pairs(VALID_LABEL_START, VALID_LABEL_END, VALID_CUTOFF, "valid")

print("train_labels:", train_labels.shape, "users:", train_labels.user_id.nunique())
print("valid_labels:", valid_labels.shape, "users:", valid_labels.user_id.nunique())
display(train_labels.head())

[COLLECT LABELS] train: 2026-02-13 -> 2026-03-12 | cutoff=2026-02-12
processed event files: 50/500 | parts=50
processed event files: 100/500 | parts=100
processed event files: 150/500 | parts=150
processed event files: 200/500 | parts=200
processed event files: 250/500 | parts=250
processed event files: 300/500 | parts=300
processed event files: 350/500 | parts=350
processed event files: 400/500 | parts=400
processed event files: 450/500 | parts=450
processed event files: 500/500 | parts=500
labels after posted_date<=cutoff filter: 3,790,119 -> 1,647,063
[SAVE] labels: /kaggle/working/positive_pairs_train_20260213_20260312.parquet
[COLLECT LABELS] valid: 2026-03-13 -> 2026-04-09 | cutoff=2026-03-12
processed event files: 50/500 | parts=50
processed event files: 100/500 | parts=100
processed event files: 150/500 | parts=150
processed event files: 200/500 | parts=200
processed event files: 250/500 | parts=250
processed event files: 300/500 | parts=300
processed event files: 350/500 | par

,user_id,item_id
0,c7a83e51965b0088caace9e16bf21209c51a357af7702e...,9bd0152b4e7c1ba36a5fc5a3458cbbbd7eaad097cef753...
1,d09504f284c0dd4d6a06f24441aa5eaa5e8e0f04af7184...,0b8e13e566f0df684e5189f44e1b2d7d011ef617929b35...
2,8279faa030fc9b9063799a4392b4c38d43e417840638e8...,afe358a26404901e81ee6ceddf675ebf9afc617d65f760...
3,d72f2abe12b2b0618be4adca16f48785d5c2ec91a106d1...,7b7321698e0f5b2317cc81adff934a6af143e0ccce6acf...
4,1fd5f0f99a224ff61539d187dd3010c654b8d53a980bf3...,7b71a4835f9be471a305e6ee46d7c7b88e55179942b59c...


In [5]:
# ============================================================
# CELL 5: BUILD USER PROFILES AT A CUTOFF
# ============================================================

def build_user_profiles(user_list, cand, cutoff, name):
    cutoff = pd.Timestamp(cutoff)
    user_list = list(map(str, user_list))
    out_path = WORK_DIR / f"user_profiles_{name}_{tag_date(cutoff)}_{len(user_list)}u.parquet"
    if out_path.exists():
        df = pd.read_parquet(out_path)
        print(f"[LOAD] profiles {name}:", df.shape, out_path)
        return df

    print("=" * 100)
    print(f"[BUILD PROFILES] {name} | cutoff={cutoff.date()} | users={len(user_list):,}")
    print("=" * 100)

    users_lf = pl.DataFrame({"user_id": user_list}).lazy()
    active_item_lf = pl.DataFrame({"item_id": cand["item_id"].astype(str).tolist()}).lazy()

    cand_feat = cand[[
        "item_id", "ad_type", "price_idx", "city_name", "district_name",
        "category", "house_type", "seller_type", "base_score"
    ]].copy()
    cand_feat["item_id"] = cand_feat["item_id"].astype(str)
    cand_feat_pl = pl.from_pandas(cand_feat).lazy()

    start_dt = (cutoff - pd.Timedelta(days=HISTORY_DAYS)).to_pydatetime()
    end_dt = (cutoff + pd.Timedelta(days=1)).to_pydatetime()

    rows = []
    for i, f in enumerate(EVENT_FILES, 1):
        try:
            part = (
                pl.scan_parquet(str(f))
                .select([
                    pl.col("user_id").cast(pl.Utf8),
                    pl.col("item_id").cast(pl.Utf8),
                    pl.col("event_type").cast(pl.Utf8),
                    pl.col("event_ts").cast(pl.Datetime, strict=False),
                ])
                .join(users_lf, on="user_id", how="semi")
                .filter((pl.col("event_ts") >= pl.lit(start_dt)) & (pl.col("event_ts") < pl.lit(end_dt)))
                .join(active_item_lf, on="item_id", how="semi")
                .join(cand_feat_pl, on="item_id", how="left")
                .with_columns([
                    pl.col("event_type").replace(EVENT_WEIGHT, default=1.0).cast(pl.Float32).alias("w"),
                ])
                .group_by(["user_id", "ad_type", "price_idx", "city_name", "district_name", "category"])
                .agg([
                    pl.sum("w").alias("total_w"),
                    pl.count().alias("n_events"),
                    pl.max("event_ts").alias("last_event_ts"),
                ])
                .collect(engine="streaming")
            )
            if part.height:
                rows.append(part)
        except Exception as e:
            print("[WARN] profile file failed:", f, repr(e))

        if i % 50 == 0:
            print(f"processed event files: {i}/{len(EVENT_FILES)} | profile parts={len(rows)}")

    if not rows:
        profiles_df = pd.DataFrame(columns=["user_id", "ad_type", "price_idx", "city_name", "district_name", "category", "total_w", "n_events"])
    else:
        prof = (
            pl.concat(rows)
            .group_by(["user_id", "ad_type", "price_idx", "city_name", "district_name", "category"])
            .agg([
                pl.sum("total_w").alias("total_w"),
                pl.sum("n_events").alias("n_events"),
                pl.max("last_event_ts").alias("last_event_ts"),
            ])
            .sort(["user_id", "total_w", "last_event_ts"], descending=[False, True, True])
            .group_by("user_id", maintain_order=True)
            .first()
        )
        profiles_df = prof.to_pandas()

    # Cleanup types.
    if len(profiles_df):
        profiles_df = ensure_str_cols(profiles_df, ["user_id", "ad_type", "city_name", "district_name"])
        profiles_df["price_idx"] = pd.to_numeric(profiles_df["price_idx"], errors="coerce").fillna(-1).astype("int16")
        profiles_df["category"] = pd.to_numeric(profiles_df["category"], errors="coerce").fillna(-1).astype("int16")
        profiles_df["total_w"] = pd.to_numeric(profiles_df["total_w"], errors="coerce").fillna(0).astype("float32")
        profiles_df["n_events"] = pd.to_numeric(profiles_df["n_events"], errors="coerce").fillna(0).astype("int16")

    profiles_df.to_parquet(out_path, index=False)
    print("[SAVE] profiles:", out_path, profiles_df.shape)
    return profiles_df

In [6]:
# ============================================================
# CELL 6: CANDIDATE RETRIEVAL + FEATURE GENERATION
# ============================================================

class CandidateEngine:
    def __init__(self, cand, profiles_df):
        self.cand = cand.sort_values("base_score", ascending=False, kind="mergesort").reset_index(drop=True).copy()
        self.item_ids = self.cand["item_id"].astype(str).to_numpy()
        self.seller_ids = self.cand["seller_id"].astype(str).to_numpy()
        self.base_scores = self.cand["base_score"].astype("float32").to_numpy()
        self.ad_arr = self.cand["ad_type"].astype(str).to_numpy()
        self.city_arr = self.cand["city_name"].astype(str).to_numpy()
        self.district_arr = self.cand["district_name"].astype(str).to_numpy()
        self.price_arr = self.cand["price_idx"].astype("int16").to_numpy()
        self.cat_arr = self.cand["category"].astype("int16").to_numpy()
        self.private_arr = (self.cand["seller_type"].astype(str).to_numpy() == "private").astype("float32")
        self.views_arr = pd.to_numeric(self.cand["views_30d"], errors="coerce").fillna(0).astype("float32").to_numpy()
        self.contacts_arr = pd.to_numeric(self.cand["contacts_30d"], errors="coerce").fillna(0).astype("float32").to_numpy()
        self.images_arr = pd.to_numeric(self.cand["images_count"], errors="coerce").fillna(0).astype("float32").to_numpy()
        self.global_indices = np.arange(len(self.cand), dtype=np.int32)
        self._build_indexes()
        self.profiles = self._profiles_dict(profiles_df)

    def _make_index(self, cols):
        idx = {}
        grouped = self.cand.groupby(cols, sort=False, observed=True).indices
        for key, inds in grouped.items():
            if not isinstance(key, tuple):
                key = (key,)
            arr = np.array(inds, dtype=np.int32)
            arr = arr[np.argsort(-self.base_scores[arr])]
            idx[key] = arr
        return idx

    def _build_indexes(self):
        self.idx_ad = self._make_index(["ad_type"])
        self.idx_ad_price = self._make_index(["ad_type", "price_idx"])
        self.idx_ad_city = self._make_index(["ad_type", "city_name"])
        self.idx_ad_city_district = self._make_index(["ad_type", "city_name", "district_name"])
        self.idx_ad_price_city = self._make_index(["ad_type", "price_idx", "city_name"])
        self.idx_ad_price_city_district = self._make_index(["ad_type", "price_idx", "city_name", "district_name"])
        self.idx_ad_category = self._make_index(["ad_type", "category"])
        self.idx_ad_price_category = self._make_index(["ad_type", "price_idx", "category"])

    def _profiles_dict(self, profiles_df):
        profiles = {}
        if profiles_df is None or len(profiles_df) == 0:
            return profiles
        for row in profiles_df.itertuples(index=False):
            ad_type = getattr(row, "ad_type")
            if ad_type not in ("let", "sell"):
                continue
            profiles[str(row.user_id)] = {
                "ad_type": str(ad_type),
                "price_idx": int(row.price_idx) if not pd.isna(row.price_idx) else -1,
                "city_name": None if pd.isna(row.city_name) else str(row.city_name),
                "district_name": None if pd.isna(row.district_name) else str(row.district_name),
                "category": int(row.category) if not pd.isna(row.category) else -1,
                "total_w": float(row.total_w) if not pd.isna(row.total_w) else 0.0,
                "n_events": int(row.n_events) if hasattr(row, "n_events") and not pd.isna(row.n_events) else 0,
            }
        return profiles

    def price_neighbors(self, ad_type, price_idx):
        if price_idx is None or price_idx < 0:
            return []
        max_idx = len(LET_PRICE_ORDER) - 1 if ad_type == "let" else len(SELL_PRICE_ORDER) - 1
        return [p for p in [price_idx - 1, price_idx, price_idx + 1] if 0 <= p <= max_idx]

    def unique_concat(self, arrays, per_array_limit=PREFILTER_PER_BUCKET, total_limit=MAX_PREFILTER_CANDIDATES):
        seen = set()
        out = []
        for arr in arrays:
            if arr is None or len(arr) == 0:
                continue
            for x in arr[:per_array_limit]:
                xi = int(x)
                if xi not in seen:
                    seen.add(xi)
                    out.append(xi)
                    if len(out) >= total_limit:
                        return np.array(out, dtype=np.int32)
        return np.array(out, dtype=np.int32)

    def retrieve_candidates(self, profile):
        if not profile:
            return self.global_indices[:MAX_PREFILTER_CANDIDATES]
        ad_type = profile["ad_type"]
        price_idx = profile.get("price_idx", -1)
        city = profile.get("city_name")
        district = profile.get("district_name")
        category = profile.get("category", -1)
        prices = self.price_neighbors(ad_type, price_idx)
        arrays = []

        if city and district and prices:
            arrays.extend(self.idx_ad_price_city_district.get((ad_type, p, city, district)) for p in prices)
        if city and prices:
            arrays.extend(self.idx_ad_price_city.get((ad_type, p, city)) for p in prices)
        if category is not None and category >= 0 and prices:
            arrays.extend(self.idx_ad_price_category.get((ad_type, p, category)) for p in prices)
        if prices:
            arrays.extend(self.idx_ad_price.get((ad_type, p)) for p in prices)
        if city and district:
            arrays.append(self.idx_ad_city_district.get((ad_type, city, district)))
        if city:
            arrays.append(self.idx_ad_city.get((ad_type, city)))
        if category is not None and category >= 0:
            arrays.append(self.idx_ad_category.get((ad_type, category)))
        arrays.append(self.idx_ad.get((ad_type,)))
        arrays.append(self.global_indices)
        return self.unique_concat(arrays)

    def rule_scores(self, indices, profile):
        scores = self.base_scores[indices].astype("float32").copy()
        if not profile:
            return scores
        price_idx = profile.get("price_idx", -1)
        if price_idx >= 0:
            dist = np.abs(self.price_arr[indices].astype("int16") - int(price_idx)).astype("float32")
            scores += np.maximum(0.0, 1.25 - 0.55 * dist)
        city = profile.get("city_name")
        if city:
            scores += (self.city_arr[indices] == city).astype("float32") * 1.20
        district = profile.get("district_name")
        if district:
            scores += (self.district_arr[indices] == district).astype("float32") * 1.60
        category = profile.get("category", -1)
        if category is not None and category >= 0:
            scores += (self.cat_arr[indices] == int(category)).astype("float32") * 0.45
        scores += self.private_arr[indices] * 0.10
        return scores

    def top_candidate_indices(self, user_id, topk=CANDIDATE_TOPK):
        profile = self.profiles.get(str(user_id))
        indices = self.retrieve_candidates(profile)
        scores = self.rule_scores(indices, profile)
        order = np.argsort(-scores, kind="mergesort")[:topk]
        return indices[order], scores[order], profile

    def make_features_for_user(self, user_id, topk=CANDIDATE_TOPK):
        indices, scores, profile = self.top_candidate_indices(user_id, topk=topk)
        if len(indices) == 0:
            return pd.DataFrame()
        ranks = np.arange(1, len(indices) + 1, dtype=np.int16)
        if profile:
            prof_ad = profile.get("ad_type", "__missing__")
            prof_price = int(profile.get("price_idx", -1))
            prof_city = profile.get("city_name") or "__missing__"
            prof_district = profile.get("district_name") or "__missing__"
            prof_cat = int(profile.get("category", -1))
            prof_w = float(profile.get("total_w", 0.0))
            prof_n = int(profile.get("n_events", 0))
        else:
            prof_ad, prof_price, prof_city, prof_district, prof_cat, prof_w, prof_n = "__cold__", -1, "__missing__", "__missing__", -1, 0.0, 0

        price_dist = np.abs(self.price_arr[indices].astype("int16") - prof_price).astype("int16") if prof_price >= 0 else np.full(len(indices), 99, dtype=np.int16)
        df = pd.DataFrame({
            "user_id": str(user_id),
            "item_id": self.item_ids[indices].astype(str),
            "cand_rank": ranks,
            "rule_score": scores.astype("float32"),
            "base_score": self.base_scores[indices].astype("float32"),
            "log_views_30d": np.log1p(self.views_arr[indices]).astype("float32"),
            "log_contacts_30d": np.log1p(self.contacts_arr[indices]).astype("float32"),
            "images_count": self.images_arr[indices].astype("float32"),
            "item_ad_type": self.ad_arr[indices].astype(str),
            "item_city": self.city_arr[indices].astype(str),
            "item_district": self.district_arr[indices].astype(str),
            "item_category": self.cat_arr[indices].astype("int16"),
            "item_price_idx": self.price_arr[indices].astype("int16"),
            "is_private": self.private_arr[indices].astype("int8"),
            "prof_ad_type": prof_ad,
            "prof_city": prof_city,
            "prof_district": prof_district,
            "prof_category": prof_cat,
            "prof_price_idx": prof_price,
            "prof_total_w": np.float32(prof_w),
            "prof_n_events": np.int16(prof_n),
            "same_ad_type": (self.ad_arr[indices] == prof_ad).astype("int8"),
            "same_city": (self.city_arr[indices] == prof_city).astype("int8"),
            "same_district": (self.district_arr[indices] == prof_district).astype("int8"),
            "same_category": (self.cat_arr[indices] == prof_cat).astype("int8"),
            "price_dist": price_dist,
        })
        return df


def build_feature_rows(user_list, engine, name, labels_df=None, topk=CANDIDATE_TOPK, max_neg_per_user=MAX_NEG_PER_USER):
    out_path = WORK_DIR / f"ranker_rows_{name}_top{topk}_neg{max_neg_per_user}.parquet"
    if out_path.exists():
        df = pd.read_parquet(out_path)
        print(f"[LOAD] ranker rows {name}:", df.shape, out_path)
        return df

    print("=" * 100)
    print(f"[BUILD RANKER ROWS] {name} | users={len(user_list):,} | topk={topk}")
    print("=" * 100)

    label_set = None
    if labels_df is not None and len(labels_df):
        label_set = set(zip(labels_df["user_id"].astype(str), labels_df["item_id"].astype(str)))

    chunks = []
    for i, u in enumerate(user_list, 1):
        df_u = engine.make_features_for_user(u, topk=topk)
        if df_u.empty:
            continue
        if label_set is not None:
            df_u["label"] = [1 if (str(u), str(it)) in label_set else 0 for it in df_u["item_id"]]
            # Keep all positives and sample negatives per user to avoid OOM.
            pos = df_u[df_u["label"] == 1]
            neg = df_u[df_u["label"] == 0]
            if max_neg_per_user is not None and len(neg) > max_neg_per_user:
                neg = neg.sample(n=max_neg_per_user, random_state=RANDOM_SEED + i)
            df_u = pd.concat([pos, neg], ignore_index=True)
        chunks.append(df_u)

        if i % 5000 == 0:
            print(f"users processed: {i:,}/{len(user_list):,} | chunks={len(chunks):,}")

    if not chunks:
        df = pd.DataFrame()
    else:
        df = pd.concat(chunks, ignore_index=True)

    if label_set is not None and len(df):
        # Drop users with no positive candidate for YetiRank training.
        pos_per_user = df.groupby("user_id")["label"].sum()
        keep_users = set(pos_per_user[pos_per_user > 0].index.astype(str))
        before = len(df)
        df = df[df["user_id"].astype(str).isin(keep_users)].copy()
        print(f"drop groups without positive candidate: {before:,} -> {len(df):,} rows | groups={len(keep_users):,}")

    df.to_parquet(out_path, index=False)
    print("[SAVE] ranker rows:", out_path, df.shape)
    return df

In [7]:
# ============================================================
# CELL 7: BUILD TRAIN / VALID DATA FOR YETIRANK
# ============================================================

# Deterministic user subsets from label users.
train_users_all = sorted(train_labels["user_id"].astype(str).unique().tolist())
valid_users_all = sorted(valid_labels["user_id"].astype(str).unique().tolist())

train_users = hash_sample_users(train_users_all, TRAIN_USER_LIMIT, seed=RANDOM_SEED)
valid_users = hash_sample_users(valid_users_all, VALID_USER_LIMIT, seed=RANDOM_SEED + 100)

print("train_users:", len(train_users), "/", len(train_users_all))
print("valid_users:", len(valid_users), "/", len(valid_users_all))

# Candidate pools are cutoff-specific to avoid time leakage.
train_cand = build_candidate_pool(TRAIN_CUTOFF, "train")
valid_cand = build_candidate_pool(VALID_CUTOFF, "valid")

train_profiles = build_user_profiles(train_users, train_cand, TRAIN_CUTOFF, "train")
valid_profiles = build_user_profiles(valid_users, valid_cand, VALID_CUTOFF, "valid")

train_engine = CandidateEngine(train_cand, train_profiles)
valid_engine = CandidateEngine(valid_cand, valid_profiles)

train_ranker_df = build_feature_rows(
    train_users,
    train_engine,
    name="train",
    labels_df=train_labels,
    topk=CANDIDATE_TOPK,
    max_neg_per_user=MAX_NEG_PER_USER,
)
valid_ranker_df = build_feature_rows(
    valid_users,
    valid_engine,
    name="valid",
    labels_df=valid_labels,
    topk=CANDIDATE_TOPK,
    max_neg_per_user=None,   # keep full top150 for honest validation
)

print("train_ranker_df:", train_ranker_df.shape)
print("valid_ranker_df:", valid_ranker_df.shape)
print("train pos rate:", train_ranker_df["label"].mean() if len(train_ranker_df) else None)
print("valid pos rate:", valid_ranker_df["label"].mean() if len(valid_ranker_df) else None)
display(train_ranker_df.head())

train_users: 120000 / 435528
valid_users: 50000 / 525420
[BUILD CANDIDATE POOL] name=train cutoff=2026-02-12
active candidate rows collected: (106924, 27)
candidate pool after property dedup: 106,924 -> 80,379
[SAVE] candidate pool: /kaggle/working/candidate_pool_train_20260212.parquet (80379, 34)
[BUILD CANDIDATE POOL] name=valid cutoff=2026-03-12
active candidate rows collected: (118701, 27)
candidate pool after property dedup: 118,701 -> 89,861
[SAVE] candidate pool: /kaggle/working/candidate_pool_valid_20260312.parquet (89861, 34)
[BUILD PROFILES] train | cutoff=2026-02-12 | users=120,000
processed event files: 50/500 | profile parts=50
processed event files: 100/500 | profile parts=100
processed event files: 150/500 | profile parts=150
processed event files: 200/500 | profile parts=200
processed event files: 250/500 | profile parts=250
processed event files: 300/500 | profile parts=300
processed event files: 350/500 | profile parts=350
processed event files: 400/500 | profile part

,user_id,item_id,cand_rank,rule_score,base_score,log_views_30d,log_contacts_30d,images_count,item_ad_type,item_city,...,prof_category,prof_price_idx,prof_total_w,prof_n_events,same_ad_type,same_city,same_district,same_category,price_dist,label
80,e54463df8533eb1e1e86d990daedb87ceb48595efd8bab...,bba5cc82760bf4f5632d5e9d88966ea9846d3faa56d978...,52,26.971031,24.071030,5.926926,4.465908,4.0,sell,Đà Nẵng,...,1040,-1,3.0,3,1,1,1,0,99,1
81,e54463df8533eb1e1e86d990daedb87ceb48595efd8bab...,8e7022c2eec5cb9f10d5204f42fa9d2b0fa5f0caa9278c...,1,33.288219,33.188221,8.723231,6.040255,7.0,let,Tp Hồ Chí Minh,...,1040,-1,3.0,3,0,0,0,0,99,0
82,e54463df8533eb1e1e86d990daedb87ceb48595efd8bab...,1e79061fc24a74a15169068440b9a204704fa801923629...,44,27.211250,27.111250,7.437206,4.844187,6.0,let,Tp Hồ Chí Minh,...,1040,-1,3.0,3,0,0,0,0,99,0
83,e54463df8533eb1e1e86d990daedb87ceb48595efd8bab...,53e8da8796f679c2485c88938b427b68c386089d7275d2...,130,25.699745,25.599745,6.981006,4.574711,10.0,let,Tp Hồ Chí Minh,...,1040,-1,3.0,3,0,0,0,0,99,0
84,e54463df8533eb1e1e86d990daedb87ceb48595efd8bab...,98c0f5e2c3c80f707085f6c4d331e336eeec666d55f625...,4,30.803925,30.703924,7.118826,5.828946,3.0,let,Tp Hồ Chí Minh,...,1040,-1,3.0,3,0,0,0,0,99,0


In [8]:
# ============================================================
# CELL 8: TRAIN CATBOOSTRANKER YETIRANK ON GPU
# ============================================================

assert CATBOOST_OK, "CatBoost is not installed/importable."
assert len(train_ranker_df) > 0, "Empty train_ranker_df. Check labels/candidates."
assert len(valid_ranker_df) > 0, "Empty valid_ranker_df. Check labels/candidates."

ID_COLS = ["user_id", "item_id"]
TARGET_COL = "label"

# CatBoost categorical features.
CAT_COLS = [
    "item_ad_type", "item_city", "item_district",
    "prof_ad_type", "prof_city", "prof_district",
]

DROP_COLS = ID_COLS + [TARGET_COL]
if DROP_RULE_SCORE_FEATURE:
    DROP_COLS += ["rule_score", "base_score"]

FEATURE_COLS = [c for c in train_ranker_df.columns if c not in DROP_COLS]

# Robust cleanup.
for df in [train_ranker_df, valid_ranker_df]:
    for c in CAT_COLS:
        if c in df.columns:
            df[c] = df[c].fillna("__missing__").astype(str)
    for c in FEATURE_COLS:
        if c not in CAT_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(-1)

cat_features = [FEATURE_COLS.index(c) for c in CAT_COLS if c in FEATURE_COLS]
print("n_features:", len(FEATURE_COLS))
print("cat_features:", [FEATURE_COLS[i] for i in cat_features])
print("dropped numeric rule score?", DROP_RULE_SCORE_FEATURE)

# Critical for ranking: rows of the same user/group must be contiguous.
train_ranker_df = train_ranker_df.sort_values(["user_id", "label", "cand_rank"], ascending=[True, False, True]).reset_index(drop=True)
valid_ranker_df = valid_ranker_df.sort_values(["user_id", "label", "cand_rank"], ascending=[True, False, True]).reset_index(drop=True)

train_pool = Pool(
    data=train_ranker_df[FEATURE_COLS],
    label=train_ranker_df[TARGET_COL].astype(float),
    group_id=train_ranker_df["user_id"].astype(str),
    cat_features=cat_features,
)
valid_pool = Pool(
    data=valid_ranker_df[FEATURE_COLS],
    label=valid_ranker_df[TARGET_COL].astype(float),
    group_id=valid_ranker_df["user_id"].astype(str),
    cat_features=cat_features,
)

params = dict(
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    task_type="GPU",
    devices="0",
    iterations=1800,
    learning_rate=0.045,
    depth=7,
    l2_leaf_reg=8.0,
    random_seed=RANDOM_SEED,
    od_type="Iter",
    od_wait=150,
    verbose=100,
    allow_writing_files=True,
)

print("[TRAIN] CatBoostRanker YetiRank GPU")
model = CatBoostRanker(**params)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
model.save_model(str(MODEL_PATH))

with open(WORK_DIR / "yetirank_feature_cols.json", "w") as f:
    json.dump({"feature_cols": FEATURE_COLS, "cat_cols": CAT_COLS, "drop_rule_score_feature": DROP_RULE_SCORE_FEATURE}, f, indent=2)

print("[SAVE] model:", MODEL_PATH)
print("best_iteration:", model.get_best_iteration())
print("best_score:", model.get_best_score())

n_features: 22
cat_features: ['item_ad_type', 'item_city', 'item_district', 'prof_ad_type', 'prof_city', 'prof_district']
dropped numeric rule score? True
[TRAIN] CatBoostRanker YetiRank GPU
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because PFound, NDCG is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.1032607	best: 0.1032607 (0)	total: 5.72s	remaining: 2h 51m 25s
100:	test: 0.1715278	best: 0.1725977 (74)	total: 11.2s	remaining: 3m 8s
200:	test: 0.1841612	best: 0.1841612 (200)	total: 16.8s	remaining: 2m 13s
300:	test: 0.1859757	best: 0.1863195 (255)	total: 22.3s	remaining: 1m 50s
400:	test: 0.1896499	best: 0.1898182 (398)	total: 27.8s	remaining: 1m 36s
500:	test: 0.1916295	best: 0.1922261 (480)	total: 33.3s	remaining: 1m 26s
600:	test: 0.1935832	best: 0.1940733 (587)	total: 38.8s	remaining: 1m 17s
700:	test: 0.1974817	best: 0.1976628 (646)	total: 44.3s	remaining: 1m 9s
800:	test: 0.1976368	best: 0.1978647 (789)	total: 49.8s	remaining: 1m 2s
900:	test: 0.1978472	best: 0.1981180 (897)	total: 55.4s	remaining: 55.2s
1000:	test: 0.1983286	best: 0.1983286 (1000)	total: 1m	remaining: 48.6s
1100:	test: 0.1980409	best: 0.1988284 (1088)	total: 1m 6s	remaining: 42.4s
1200:	test: 0.1983855	best: 0.1988284 (1088)	total: 1m 12s	remaining: 36.1s
bestTest = 0.198828351
bestIteration = 108

In [9]:
# ============================================================
# CELL 9: VALIDATION METRICS — RECALL@10 / NDCG@10
# ============================================================

def recall_ndcg_at_k(df, score_col="pred", k=10):
    # df must contain user_id, item_id, label, pred.
    hit_pairs = 0
    total_pairs = int(df["label"].sum())
    hit_users = 0
    total_users = df["user_id"].nunique()
    ndcg_sum = 0.0

    for u, g in df.groupby("user_id", sort=False):
        g = g.sort_values(score_col, ascending=False, kind="mergesort").head(k)
        labels = g["label"].to_numpy(dtype=np.int8)
        hits = int(labels.sum())
        hit_pairs += hits
        if hits > 0:
            hit_users += 1
        # DCG with binary relevance.
        dcg = 0.0
        for i, rel in enumerate(labels, start=1):
            if rel:
                dcg += 1.0 / math.log2(i + 1)
        # ideal DCG capped at min(number positives in candidate set, k)
        n_pos_user = int(df.loc[df["user_id"].eq(u), "label"].sum())
        ideal_hits = min(n_pos_user, k)
        idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
        if idcg > 0:
            ndcg_sum += dcg / idcg

    return {
        "k": k,
        "hit_pairs": hit_pairs,
        "total_positive_candidate_pairs": total_pairs,
        "pair_recall_at_k": hit_pairs / max(total_pairs, 1),
        "hit_users": hit_users,
        "total_users": total_users,
        "hit_user_rate_at_k": hit_users / max(total_users, 1),
        "mean_ndcg_at_k": ndcg_sum / max(total_users, 1),
    }

valid_ranker_df["pred"] = model.predict(valid_pool)

metrics = []
for k in [10, 20, 50, 100, 150]:
    metrics.append(recall_ndcg_at_k(valid_ranker_df, "pred", k=k))
metrics_df = pd.DataFrame(metrics)
display(metrics_df)

# Compare with candidate order/rule retrieval rank, to verify CatBoost is not worse than rule popularity rank.
valid_ranker_df["rule_rank_score"] = -valid_ranker_df["cand_rank"].astype(float)
rule_metrics = []
for k in [10, 20, 50, 100, 150]:
    rule_metrics.append(recall_ndcg_at_k(valid_ranker_df, "rule_rank_score", k=k))
rule_metrics_df = pd.DataFrame(rule_metrics)
print("[RULE/CANDIDATE ORDER BASELINE]")
display(rule_metrics_df)

,k,hit_pairs,total_positive_candidate_pairs,pair_recall_at_k,hit_users,total_users,hit_user_rate_at_k,mean_ndcg_at_k
0,10,1190,3326,0.357787,1065,2792,0.381447,0.198828
1,20,1675,3326,0.503608,1446,2792,0.517908,0.235807
2,50,2245,3326,0.674985,1887,2792,0.675860,0.269697
3,100,2970,3326,0.892965,2486,2792,0.890401,0.307607
4,150,3326,3326,1.000000,2792,2792,1.000000,0.324725


[RULE/CANDIDATE ORDER BASELINE]


,k,hit_pairs,total_positive_candidate_pairs,pair_recall_at_k,hit_users,total_users,hit_user_rate_at_k,mean_ndcg_at_k
0,10,537,3326,0.161455,514,2792,0.184097,0.078548
1,20,732,3326,0.220084,685,2792,0.245344,0.093664
2,50,1405,3326,0.422429,1275,2792,0.456662,0.135223
3,100,2386,3326,0.717378,2057,2792,0.736748,0.184349
4,150,3326,3326,1.000000,2792,2792,1.000000,0.227324


In [10]:
# ============================================================
# CELL 10: FINAL PREDICT — WARM USER YETIRANK, COLD USER SUBMISSION2
# ============================================================

# Build final candidate pool/profile using all test users at final cutoff.
final_cand = build_candidate_pool(FINAL_CUTOFF, "final")
final_profiles = build_user_profiles(test_user_list, final_cand, FINAL_CUTOFF, "final_test")
final_engine = CandidateEngine(final_cand, final_profiles)

warm_users = set(final_engine.profiles.keys())
print("warm_users with history profile:", len(warm_users))
print("cold_users:", len(test_user_list) - len(warm_users))

# Load strong fallback submission2.
fallback_map = {}
if SUBMISSION2_PATH is not None:
    fb = pd.read_csv(SUBMISSION2_PATH)
    # Support either ID,user_id,rank,item_id or user_id,rank,item_id.
    fb["user_id"] = fb["user_id"].astype(str)
    fb["item_id"] = fb["item_id"].astype(str)
    fb = fb.sort_values(["user_id", "rank"], kind="mergesort")
    for u, g in fb.groupby("user_id", sort=False):
        fallback_map[str(u)] = g["item_id"].head(TOPN).tolist()
    print("fallback users loaded:", len(fallback_map))
else:
    # Global fallback from final candidate pool.
    global_items = final_cand["item_id"].astype(str).head(500).tolist()
    for u in test_user_list:
        fallback_map[str(u)] = global_items[:TOPN]

# Feature schema reload in case notebook restarted.
feature_meta_path = WORK_DIR / "yetirank_feature_cols.json"
if feature_meta_path.exists():
    meta = json.load(open(feature_meta_path))
    FEATURE_COLS = meta["feature_cols"]
    CAT_COLS = meta["cat_cols"]

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}. Run Cell 8 first.")

# If model object does not exist in memory, load it.
try:
    model
except NameError:
    model = CatBoostRanker()
    model.load_model(str(MODEL_PATH))


def prepare_pred_features(df):
    for c in CAT_COLS:
        if c in df.columns:
            df[c] = df[c].fillna("__missing__").astype(str)
    for c in FEATURE_COLS:
        if c not in df.columns:
            df[c] = -1
        if c not in CAT_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(-1)
    return df


def select_top10_with_seller_cap(df_scores, final_cand_lookup):
    # df_scores: item_id, pred sorted descending later.
    g = df_scores.sort_values("pred", ascending=False, kind="mergesort")
    selected = []
    seller_count = Counter()
    seen = set()
    for row in g.itertuples(index=False):
        item = str(row.item_id)
        if item in seen:
            continue
        seller = final_cand_lookup.get(item, "__missing__")
        if seller_count[seller] >= SELLER_CAP:
            continue
        selected.append(item)
        seen.add(item)
        seller_count[seller] += 1
        if len(selected) >= TOPN:
            break
    return selected

seller_lookup = dict(zip(final_cand["item_id"].astype(str), final_cand["seller_id"].astype(str)))
global_final_items = final_cand["item_id"].astype(str).head(1000).tolist()

with open(FINAL_OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["ID", "user_id", "rank", "item_id"])
    row_id = 1

    batch_rows = []
    batch_users = []
    BATCH_WARM_USERS = 3000

    def flush_batch(batch_rows, batch_users, row_id):
        if not batch_users:
            return row_id
        pred_df = pd.concat(batch_rows, ignore_index=True)
        pred_df = prepare_pred_features(pred_df)
        pool = Pool(pred_df[FEATURE_COLS], cat_features=[FEATURE_COLS.index(c) for c in CAT_COLS if c in FEATURE_COLS])
        pred_df["pred"] = model.predict(pool)

        pred_map = {}
        for u, g in pred_df.groupby("user_id", sort=False):
            pred_map[str(u)] = select_top10_with_seller_cap(g[["item_id", "pred"]], seller_lookup)

        for u in batch_users:
            recs = pred_map.get(str(u), [])
            # fill from fallback then global if needed
            for item in fallback_map.get(str(u), []):
                if len(recs) >= TOPN:
                    break
                if item not in recs:
                    recs.append(item)
            for item in global_final_items:
                if len(recs) >= TOPN:
                    break
                if item not in recs:
                    recs.append(item)
            recs = recs[:TOPN]
            for rank, item in enumerate(recs, 1):
                writer.writerow([row_id, str(u), rank, str(item)])
                row_id += 1
        return row_id

    for i, u in enumerate(test_user_list, 1):
        su = str(u)
        if su in warm_users:
            df_u = final_engine.make_features_for_user(su, topk=CANDIDATE_TOPK)
            if len(df_u):
                batch_rows.append(df_u)
                batch_users.append(su)
            else:
                recs = fallback_map.get(su, global_final_items[:TOPN])[:TOPN]
                for rank, item in enumerate(recs, 1):
                    writer.writerow([row_id, su, rank, str(item)])
                    row_id += 1
        else:
            recs = fallback_map.get(su, global_final_items[:TOPN])[:TOPN]
            # safety fill
            for item in global_final_items:
                if len(recs) >= TOPN:
                    break
                if item not in recs:
                    recs.append(item)
            for rank, item in enumerate(recs[:TOPN], 1):
                writer.writerow([row_id, su, rank, str(item)])
                row_id += 1

        if len(batch_users) >= BATCH_WARM_USERS:
            row_id = flush_batch(batch_rows, batch_users, row_id)
            batch_rows, batch_users = [], []
            gc.collect()

        if i % 10000 == 0:
            print(f"processed final users: {i:,}/{len(test_user_list):,} | rows={row_id-1:,}")

    row_id = flush_batch(batch_rows, batch_users, row_id)

print("[DONE] final submission:", FINAL_OUTPUT_PATH)
print("rows written:", row_id - 1)

[BUILD CANDIDATE POOL] name=final cutoff=2026-04-09
active candidate rows collected: (141260, 27)
candidate pool after property dedup: 141,260 -> 107,711
[SAVE] candidate pool: /kaggle/working/candidate_pool_final_20260409.parquet (107711, 34)
[BUILD PROFILES] final_test | cutoff=2026-04-09 | users=161,568
processed event files: 50/500 | profile parts=50
processed event files: 100/500 | profile parts=100
processed event files: 150/500 | profile parts=150
processed event files: 200/500 | profile parts=200
processed event files: 250/500 | profile parts=250
processed event files: 300/500 | profile parts=300
processed event files: 350/500 | profile parts=350
processed event files: 400/500 | profile parts=400
processed event files: 450/500 | profile parts=450
processed event files: 500/500 | profile parts=500
[SAVE] profiles: /kaggle/working/user_profiles_final_test_20260409_161568u.parquet (54229, 9)
warm_users with history profile: 54229
cold_users: 107339
fallback users loaded: 161568
pr

In [11]:
# ============================================================
# CELL 11: SUBMISSION SANITY CHECK
# ============================================================

sub = pl.scan_csv(str(FINAL_OUTPUT_PATH), infer_schema_length=1000)
summary = sub.select([
    pl.len().alias("n_rows"),
    pl.n_unique("user_id").alias("n_users"),
    pl.min("ID").alias("min_id"),
    pl.max("ID").alias("max_id"),
    pl.min("rank").alias("min_rank"),
    pl.max("rank").alias("max_rank"),
]).collect(engine="streaming")
display(summary)

assert summary["n_rows"][0] == len(test_user_list) * TOPN
assert summary["n_users"][0] == len(test_user_list)
assert summary["min_id"][0] == 1
assert summary["max_id"][0] == len(test_user_list) * TOPN
assert summary["min_rank"][0] == 1
assert summary["max_rank"][0] == TOPN

rank1_users = (
    sub
    .filter(pl.col("rank") == 1)
    .select("user_id")
    .collect(engine="streaming")["user_id"]
    .to_list()
)
assert rank1_users == test_user_list, "user_id order mismatch versus test_users.parquet"

# Validate all item IDs exist in official dim_listing.
official_items = (
    pl.scan_parquet([str(p) for p in DIM_FILES])
    .select(pl.col("item_id").cast(pl.Utf8))
    .unique()
    .collect(engine="streaming")
)
invalid = (
    sub
    .select(pl.col("item_id").cast(pl.Utf8))
    .join(official_items.lazy(), on="item_id", how="anti")
    .collect(engine="streaming")
)
print("invalid item rows:", invalid.height)
assert invalid.height == 0, "submission contains invalid item_id"

# Duplicate per user check.
dup = (
    sub
    .group_by(["user_id", "item_id"])
    .agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)
    .collect(engine="streaming")
)
print("duplicate user-item rows:", dup.height)
assert dup.height == 0, "duplicate item_id inside some user top10"

print("READY:", FINAL_OUTPUT_PATH)

n_rows,n_users,min_id,max_id,min_rank,max_rank
u32,u32,i64,i64,i64,i64
1615680,161568,1,1615680,1,10


AssertionError: user_id order mismatch versus test_users.parquet